# 🤖 Webinar S10.1 – ¿Qué es Machine Learning?
**Curso de Data Science & Machine Learning**  
Dataset: Online Retail (ventas de e-commerce UK)

---
## Índice
1. Setup e importaciones
2. Exploración del dataset
3. ¿Predecir o describir? – primeras preguntas de negocio
4. Preparación de variables: vectorización
5. Primer modelo con sklearn
6. 🧪 Ejercicios

---
## 1. Setup e Importaciones

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("✅ Librerías cargadas correctamente")

---
## 2. Exploración del Dataset

Usaremos el **Online Retail Dataset** – transacciones de una tienda de e-commerce del Reino Unido (2010-2011).  
Cada fila representa una línea de una factura.

**Variables disponibles:**
| Columna | Descripción |
|---|---|
| `InvoiceNo` | Número de factura |
| `StockCode` | Código del producto |
| `Description` | Descripción del producto |
| `Quantity` | Unidades compradas |
| `InvoiceDate` | Fecha de la transacción |
| `UnitPrice` | Precio unitario (GBP) |
| `CustomerID` | ID del cliente |
| `Country` | País del cliente |

In [ ]:
# Cargamos el dataset 
df = pd.read_excel('online_retail.xlsx',nrows=100000)
df.head()

In [ ]:
df.info()

In [ ]:
# Estadísticas básicas
df.describe()

In [ ]:
# ¿Cuántos clientes únicos hay?
print(f"Clientes únicos: {df['CustomerID'].nunique()}")
print(f"Países: {df['Country'].unique()}")
print(f"\nDistribución por país:")
print(df['Country'].value_counts())

---
## 3. ¿Predecir o Describir? – Preguntas de Negocio

Antes de escribir una línea de código de ML, siempre pregúntate:  
**¿Qué quiero saber? ¿Es una pregunta de predicción o de descripción?**

| Pregunta | Tipo | Tarea ML |
|---|---|---|
| ¿Este cliente volverá a comprar? | Predecir | Clasificación |
| ¿Cuánto gastará un cliente el próximo mes? | Predecir | Regresión |
| ¿Qué grupos de clientes similares existen? | Describir | Clustering (no supervisado) |
| ¿Qué productos se compran juntos? | Describir | Reglas de asociación |

En este webinar haremos una tarea de **clasificación**: predecir si un cliente es "alto valor" o no.

In [ ]:
# Creamos nuestro target: ¿es un cliente de alto valor?
# Definimos "alto valor" como clientes que gastan más de la mediana

df["TotalSpent"] = df['UnitPrice'] * df['Quantity']

# Agrupamos por cliente y calculamos gasto total
clientes = df.groupby('CustomerID').agg(
    total_gastado=('TotalSpent', 'sum'),
    num_compras=('Quantity', 'count'),
    pais=('Country', 'first')
).reset_index()

mediana = clientes['total_gastado'].median()
print(f"Mediana de gasto por cliente: £{mediana:.2f}")

clientes.head()

---
## 4. Preparación de Variables: Vectorización

> **Regla de oro:** Siempre que puedas, trabaja con operaciones vectorizadas (sobre columnas completas)  
> Evita los `for` loops sobre filas — son lentos con DataFrames grandes.

### 4.1 Mapeo Binario

In [ ]:
# ✅ Vectorizado con np.where – RÁPIDO
clientes['es_alto_valor'] = np.where(clientes['total_gastado'] > mediana, 1, 0)

# ✅ Forma RÁPIDA 2.0
#clientes['es_alto_valor'] = (clientes['total_gastado'] > mediana).astype(int)

print(f"Distribución del target:")
print(clientes['es_alto_valor'].value_counts())
print(f"\n% Alto valor: {clientes['es_alto_valor'].mean():.1%}")




In [ ]:
# ❌ Forma LENTA – evítala (solo para entender la diferencia)
# for i in range(len(clientes)):
#     if clientes.loc[i, 'total_gastado'] > mediana:
#         clientes.loc[i, 'es_alto_valor'] = 1
#     else:
#         clientes.loc[i, 'es_alto_valor'] = 0

# La diferencia en un DataFrame de 1M filas puede ser: 0.001s vs 3s con pocos datos

### 4.2 Mapeo de Variable Categórica (Multiclase con `np.select`)

In [ ]:
# Segmentar clientes por nivel de compras con np.select
condiciones = [
    clientes['num_compras'] < 5,
    clientes['num_compras'].between(5, 15),
    clientes['num_compras'] > 15
]
valores = ['ocasional', 'regular', 'frecuente']

clientes['segmento'] = np.select(condiciones, valores, default='desconocido')

print(clientes['segmento'].value_counts())

---
## 5. Primer Modelo con sklearn

Flujo básico:
1. Definir **features** (X) y **target** (y)
2. **Dividir** en entrenamiento y prueba con `train_test_split`
3. **Entrenar** el modelo con `.fit()`
4. **Predecir** con `.predict()`
5. **Evaluar** con una métrica

In [ ]:
# 1. Definir features y target
X = clientes[['num_compras', 'es_uk']]   # features (variables de entrada)
y = clientes['es_alto_valor']              # target (lo que queremos predecir)

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")

In [ ]:
# 2. División train/test
# random_state=42 → garantiza reproducibilidad (misma división siempre)
# test_size=0.2  → 20% para prueba, 80% para entrenamiento

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} filas")
print(f"Prueba:        {X_test.shape[0]} filas")

In [ ]:
# 3 y 4. Entrenar y predecir
modelo = LogisticRegression(random_state=42)
modelo.fit(X_train, y_train)              # ← aquí el modelo APRENDE

predicciones = modelo.predict(X_test)     # ← aquí el modelo PREDICE

print("Primeras 10 predicciones:", predicciones[:10])
print("Valores reales:          ", y_test.values[:10])

In [ ]:
# 5. Evaluar – accuracy (la veremos a fondo en el Webinar 2)
accuracy = accuracy_score(y_test, predicciones)
print(f"Accuracy del modelo: {accuracy:.2%}")
print("\n(En el próximo webinar entenderemos si esto es bueno o malo 👀)")

---
## 6. 🧪 Ejercicios

Completa las celdas marcadas con `# TU CÓDIGO AQUÍ`

### Ejercicio 1 – Mapeo Binario
Crea una columna `es_frecuente` que valga `1` si el cliente tiene más de 10 compras y `0` si no.

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2 – Mapeo Multiclase
Crea una columna `nivel_gasto` usando `np.select` con tres categorías:
- `'bajo'` si `total_gastado < 100`
- `'medio'` si `total_gastado` está entre 100 y 500
- `'alto'` si `total_gastado > 500`

In [ ]:
# TU CÓDIGO AQUÍ

---
## ✅ Resumen del Webinar 1

| Concepto | Clave |
|---|---|
| ML supervisado | Aprende de ejemplos etiquetados (X → y) |
| Clasificación vs Regresión | ¿La respuesta es una etiqueta o un número? |
| Vectorización | Siempre opera sobre columnas, nunca con for loops |
| Flujo sklearn | `split → fit → predict → evaluate` |
| `random_state` | Garantiza reproducibilidad |

**¿Qué sigue? → Webinar 2: métricas de clasificación y construcción paso a paso de un modelo completo.**